# 

In [9]:

config = { "vocab_size": 50257,   # Vocabulary size
    "context_len": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "heads": 12,         # Number of attention heads
    "tranf_blocks": 12,        # Number of layers
    "dropout_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

In [1]:
import torch
import torch.nn as nn
import tiktoken

In [2]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, context_len, stride):
        self.input_ids = [] 
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids)>context_len, "Number of tokenized input"

        for i in range(0, len(token_ids) - context_len, stride):
            input_chunks = token_ids[i:i+context_len]
            target_chunks = token_ids[i+1: i+context_len+1]

            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        
         

In [3]:
def create_dataloader(txt, bs=4, max_len=256, stride=128, 
                      shuffle=True, drop_last=True, 
                      num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDataset(txt, tokenizer, max_len, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    ) 

    return dataloader


In [4]:
with open("chats_data.txt") as f:
    raw = f.read()

small_text = raw[:20000]
dataloader = create_dataloader(small_text, 4)

In [35]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, heads, dropout, context_len, qkv_cache=False):
        super().__init__()
        self.heads = heads
        self.head_dim = emb_dim//heads
        
        self.Wk = nn.Linear(emb_dim, emb_dim)
        self.Wq = nn.Linear(emb_dim, emb_dim)
        self.Wv = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_len, context_len), 1))

        self.out_proj = nn.Linear(emb_dim, emb_dim)

    def forward(self, x):
        B, T, C = x.shape
        print("In multi head attention: ", x.shape)

        query = self.Wq(x).reshape(B, T, self.heads, self.head_dim)
        query = query.transpose(1,2) # B, H, T, D
        
        
        keys = self.Wk(x).reshape(B, T, self.heads, self.head_dim)
        keys = keys.transpose(1,2) # B, H, T, D
        
        values = self.Wv(x).reshape(B, T, self.heads, self.head_dim)
        values = values.transpose(1,2) # B, H, T, D


        attn_scores = query @ keys.transpose(-1,-2) # B, H, T, T
        mask = self.mask[:T, :T].bool()
        
        print("attn_scores shape: ", attn_scores.shape)
        print("mask shape ", mask.shape)
        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights= torch.softmax(attn_scores/values.shape[-1]**0.5, dim=-1)
        
        attn_weights = self.dropout(attn_weights)

        context_vect = (attn_weights @ values).transpose(1,2).reshape(B, T, C)

        return self.out_proj(context_vect)

In [1]:
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, heads, dropout_rate, context_len):
        super().__init__()
        self.ffn = FeedForwardN(emb_dim)
        self.mha = MultiHeadAttention(emb_dim, heads, dropout_rate, context_len, False)
        self.drop_embeds = nn.Dropout(dropout_rate)
        self.norm1 = LayerNorm(emb_dim) # need to define this
        self.norm2 = LayerNorm(emb_dim) #need to define this


    def forward(self, x):
        shortcut = x  
        x = self.norm1(x)
        x = self.mha(x)
        x = self.drop_embeds(x)
        x = x+shortcut 

        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.drop_embeds(x)
        x = x + shortcut 

        return x


class FeedForwardN(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.l1 = nn.Linear(emb_dim,  emb_dim*4)
        self.gelu = nn.GELU()
        self.l2 = nn.Linear(emb_dim*4, emb_dim)

    def forward(self, x):
        x = self.l1(x)
        x = self.gelu(x)
        x = self.l2(x)
        return x

class LayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.shift = nn.Parameter(torch.zeros(dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdims=True)
        var = torch.var(x, correction=0, keepdims=True)

        x_norm = (x-mean)/var+self.eps
        return (x_norm*self.scale) + self.shift
        
class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_len"], config["emb_dim"])
        self.drop_embeds = nn.Dropout(config["dropout_rate"])
        
        self.traf_blocks = nn.Sequential(*[TransformerBlock(config["emb_dim"], config["heads"], config["dropout_rate"], config["context_len"]) for _ in range(config["tranf_blocks"])])
        
        self.final_norm = LayerNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"])

    def forward(self, Ints):
        bs, seq_len = Ints.shape
        tok_embeds = self.tok_emb(Ints)
        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = tok_embeds + pos_embeds 
        x = self.drop_embeds(x)
        print("Going to Traf Block:", x.shape)
        x = self.traf_blocks(x)

        x= self.final_norm(x)
        return self.out_head(x)   

In [7]:
it = iter(dataloader)
trgts, data= next(it)

In [38]:
model = GPTModel(config)

In [39]:
def generate_text(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        print("idx shape: ",idx.shape, "context_len: ", context_size)
        idx_cond = idx[:, -context_size:]
        print("idx_cond shape: ", idx_cond.shape)

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        probabs = torch.softmax(logits, dim=-1)

        idx_next = torch.argmax(probabs, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

In [12]:
s_data = data[0].reshape(1,-1)
s_data.shape

torch.Size([1, 256])

In [40]:
generate_text(model, s_data, 4, 256)

idx shape:  torch.Size([1, 256]) context_len:  256
idx_cond shape:  torch.Size([1, 256])
Going to Traf Block: torch.Size([1, 256, 768])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 256, 768])
attn_scores shape:  torch.Size([1, 12, 256, 256])
mask shape  torch.Size([256, 256])
In multi head attention:  torch.Size([1, 2

tensor([[34970,    25,   257,   499,   308,  9869,   257,  5488,   198, 25918,
         25557,   296,  5549,    25,   843,   450,  5303,   275,  1219,   313,
           442,  4265,   473, 25996,   387,    72,    13,   198,    56, 10680,
           327,   325,   513,    25, 50169,   235, 41840,   235,   198, 25918,
         25557,   296,  5549,    25,   371,  4594,   502,    72,   289,  2049,
           275, 44488,    13,   198, 25918, 25557,   296,  5549,    25, 20529,
          4207,  3235,  1000,   479,   283,   256, 12236,   387,    72,  4017,
            64,   371,  3378, 26615,  2082,  6081,    13,   198, 22125,    56,
          1581,   402,    83,  2545, 31597, 34970,    25, 20849,   275, 44488,
           198,    39,  5406, 35167,  9634,   402,    83,  2545,    25,  2488,
            24, 23451,  2231,  5824,  1959,  3829, 11609,  8246, 11532,   387,
            72,    11,   318,  7204,  6324,   502, 10385,   479, 11493,  5633,
           198, 22125,    56,  1581,   402,    83,  

In [41]:
start_context = "hello i am"
tokenizer = tiktoken.get_encoding("gpt2")
encode = tokenizer.encode(start_context)
print(encode)

[31373, 1312, 716]


In [42]:
encode_tensor = torch.tensor(encode).unsqueeze(0)
encode_tensor.shape

torch.Size([1, 3])

In [24]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_embeds): Dropout(p=0.1, inplace=False)
  (traf_blocks): Sequential(
    (0): TransformerBlock(
      (ffn): FeedForwardN(
        (l1): Linear(in_features=768, out_features=3072, bias=True)
        (gelu): GELU(approximate='none')
        (l2): Linear(in_features=3072, out_features=768, bias=True)
      )
      (mha): MultiHeadAttention(
        (Wk): Linear(in_features=768, out_features=768, bias=True)
        (Wq): Linear(in_features=768, out_features=768, bias=True)
        (Wv): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (drop_embeds): Dropout(p=0.1, inplace=False)
      (norm1): LayerNorm()
      (norm2): LayerNorm()
    )
    (1): TransformerBlock(
      (ffn): FeedForwardN(
        (l1): Linear(in_features=768, out_features=3072, bias=True)
        (g

In [43]:
out = generate_text(model, encode_tensor, 6, config["context_len"])
print("model out: ", out)


idx shape:  torch.Size([1, 3]) context_len:  256
idx_cond shape:  torch.Size([1, 3])
Going to Traf Block: torch.Size([1, 3, 768])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  torch.Size([3, 3])
In multi head attention:  torch.Size([1, 3, 768])
attn_scores shape:  torch.Size([1, 12, 3, 3])
mask shape  

In [44]:
decode_text = tokenizer.decode(out.squeeze(0).tolist())
decode_text

'hello i am Fantasticirteen Amid assumption KINGWra'

In [33]:
encode_tensor

tensor([[31373,  1312,   716]])